In [1]:
import os
import re
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("../data/slm_eval_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20, 2)


,input,expected_risk
0,License: MIT License Family: Permissive Projec...,Low
1,License: MIT License Family: Permissive Projec...,Low
2,License: Apache-2.0 License Family: Permissive...,Low
3,License: BSD-3-Clause License Family: Permissi...,Low
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium


In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully.")
print("CUDA available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully.
CUDA available: True


In [4]:
def build_prompt(scenario):
    return f"""
You are an open-source license compliance assistant.

Classify the compliance risk for the scenario below.

{scenario}

Return ONLY in this format:
Risk: Low/Medium/High
Reason: one short sentence
"""

In [5]:
def predict_risk(scenario):
    prompt = build_prompt(scenario)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [6]:
def extract_risk(response):
    response_lower = response.lower()

    if "risk: high" in response_lower or re.search(r"\bhigh\b", response_lower):
        return "High"
    elif "risk: medium" in response_lower or re.search(r"\bmedium\b", response_lower):
        return "Medium"
    elif "risk: low" in response_lower or re.search(r"\blow\b", response_lower):
        return "Low"
    else:
        return "Unknown"

In [7]:
responses = []
predicted_risks = []

for idx, row in df.iterrows():
    print(f"Processing row {idx + 1}/{len(df)}")

    response = predict_risk(row["input"])
    predicted_risk = extract_risk(response)

    responses.append(response)
    predicted_risks.append(predicted_risk)

df["qwen_response"] = responses
df["predicted_risk"] = predicted_risks

df.head()

Processing row 1/20
Processing row 2/20
Processing row 3/20
Processing row 4/20
Processing row 5/20
Processing row 6/20
Processing row 7/20
Processing row 8/20
Processing row 9/20
Processing row 10/20
Processing row 11/20
Processing row 12/20
Processing row 13/20
Processing row 14/20
Processing row 15/20
Processing row 16/20
Processing row 17/20
Processing row 18/20
Processing row 19/20
Processing row 20/20


,input,expected_risk,qwen_response,predicted_risk
0,License: MIT License Family: Permissive Projec...,Low,Risk: Medium\nReason: The project is classifie...,Medium
1,License: MIT License Family: Permissive Projec...,Low,Risk: Medium\nReason: The project is commercia...,Medium
2,License: Apache-2.0 License Family: Permissive...,Low,Risk: Medium\nReason: The Apache-2.0 license i...,Medium
3,License: BSD-3-Clause License Family: Permissi...,Low,Risk: Medium\nReason: The project is a scienti...,Medium
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium,Risk: Medium\nReason: The project is licensed ...,Medium


In [8]:
y_true = df["expected_risk"]
y_pred = df["predicted_risk"]

accuracy = accuracy_score(y_true, y_pred)

print("Qwen Base SLM Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Qwen Base SLM Accuracy: 0.25

Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         7
         Low       0.00      0.00      0.00         7
      Medium       0.26      0.83      0.40         6

    accuracy                           0.25        20
   macro avg       0.09      0.28      0.13        20
weighted avg       0.08      0.25      0.12        20


Confusion Matrix:
[[0 0 7]
 [0 0 7]
 [1 0 5]]


/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [9]:
os.makedirs("../outputs", exist_ok=True)

df.to_csv("../outputs/qwen_base_slm_predictions_initial.csv", index=False)

with open("../outputs/qwen_base_slm_report_initial.txt", "w") as f:
    f.write(f"Qwen Base SLM Accuracy: {accuracy}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, y_pred))
    f.write("\nConfusion Matrix:\n")
    f.write(str(confusion_matrix(y_true, y_pred)))

print("Qwen evaluation outputs saved successfully.")

Qwen evaluation outputs saved successfully.


/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sankvin/projects/oss-compliance-ai/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war